# Sprint 5 — LightGBM, XGBoost y Optimización de Hiperparámetros (v2)
## Proyecto: Productividad Asesores de Negocios

---

### Cambios respecto a v1

- `num_class=3` (ALTO, BAJO, MEDIO)
- Variables de evento correctamente calculadas (Sprint 2 v3)
- Piso de métricas actualizado: LR baseline macro F1 = 0.7107
- Objetivo: superar 0.74 en macro F1


## 1. Configuración y carga

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import mlflow
import mlflow.lightgbm
import mlflow.xgboost
import optuna
import lightgbm as lgb
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    f1_score, cohen_kappa_score, roc_auc_score, accuracy_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix, make_scorer,
)
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

PROCESSED  = Path('../data/processed')
FIG_DIR    = Path('../reports/figures')
MLRUNS_DIR = Path('../mlruns')
FIG_DIR.mkdir(parents=True, exist_ok=True)

data          = np.load(PROCESSED / 'features_processed.npz', allow_pickle=True)
X_train       = data['X_train']
X_test        = data['X_test']
y_train       = data['y_train']
y_test        = data['y_test']
label_encoder = joblib.load(PROCESSED / 'label_encoder.joblib')
CLASS_NAMES   = list(label_encoder.classes_)
N_CLASSES     = len(CLASS_NAMES)

DB_PATH_MLFLOW = Path('../mlruns/mlflow.db')
mlflow.set_tracking_uri(f'sqlite:///{DB_PATH_MLFLOW.resolve()}')
mlflow.set_experiment('advisor_risk_classification')

F1_BASELINE = 0.7107  # LR baseline v2

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'Clases : {CLASS_NAMES} ({N_CLASSES} clases)')
print(f'Baseline a superar: {F1_BASELINE} (objetivo > 0.74)')


X_train: (1603, 22)
X_test : (401, 22)
Clases : ['ALTO', 'BAJO', 'MEDIO'] (3 clases)
Baseline a superar: 0.7107 (objetivo > 0.74)


## 2. Funciones compartidas

In [2]:
def calcular_metricas(y_true, y_pred, y_prob, class_names):
    metricas = {
        'macro_f1' : f1_score(y_true, y_pred, average='macro'),
        'kappa'    : cohen_kappa_score(y_true, y_pred),
        'auc_ovr'  : roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro'),
        'accuracy' : accuracy_score(y_true, y_pred),
    }
    f1_por_clase = f1_score(y_true, y_pred, average=None)
    for i, cls in enumerate(class_names):
        metricas[f'f1_{cls}'] = f1_por_clase[i]
    return metricas


def plot_confusion_matrix(y_true, y_pred, class_names, titulo, filepath):
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(7, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='.2f')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()


cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorer = make_scorer(f1_score, average='macro')
print('Funciones definidas.')


Funciones definidas.


## 3. LightGBM — Optimización con Optuna

In [3]:
def objective_lgbm(trial):
    params = {
        'objective'        : 'multiclass',
        'num_class'        : N_CLASSES,
        'metric'           : 'multi_logloss',
        'verbosity'        : -1,
        'random_state'     : 42,
        'is_unbalance'     : True,
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 800),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves'       : trial.suggest_int('num_leaves', 15, 63),
        'max_depth'        : trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
    }
    model  = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=scorer, n_jobs=-1)
    return scores.mean()


print('Ejecutando Optuna para LightGBM (50 trials)...')
study_lgbm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f'\nMejor macro F1 CV (LightGBM): {study_lgbm.best_value:.4f}')
print('Mejores hiperparametros:')
for k, v in study_lgbm.best_params.items():
    print(f'  {k:<25}: {v}')


Ejecutando Optuna para LightGBM (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Mejor macro F1 CV (LightGBM): 0.7057
Mejores hiperparametros:
  n_estimators             : 348
  learning_rate            : 0.02467416625153153
  num_leaves               : 42
  max_depth                : 3
  min_child_samples        : 43
  subsample                : 0.6880860278981751
  colsample_bytree         : 0.9993986750197682
  reg_alpha                : 1.2158567419926094
  reg_lambda               : 0.0001711384849837762


## 4. LightGBM — Entrenamiento final y MLflow

In [4]:
best_lgbm = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=N_CLASSES,
    verbosity=-1,
    random_state=42,
    is_unbalance=True,
    **study_lgbm.best_params
)

with mlflow.start_run(run_name='lightgbm_optuna_v2') as run_lgbm:
    mlflow.log_params({
        'model_type'   : 'LightGBM',
        'n_classes'    : N_CLASSES,
        'is_unbalance' : True,
        'optuna_trials': 50,
        'target'       : 'TARGET_B_3clases',
        **study_lgbm.best_params
    })
    mlflow.log_metric('cv_macro_f1_best', study_lgbm.best_value)

    best_lgbm.fit(X_train, y_train)
    y_pred_lgbm = best_lgbm.predict(X_test)
    y_prob_lgbm = best_lgbm.predict_proba(X_test)

    metricas_lgbm = calcular_metricas(y_test, y_pred_lgbm, y_prob_lgbm, CLASS_NAMES)
    mlflow.log_metrics({f'test_{k}': v for k, v in metricas_lgbm.items()})

    cm_path = FIG_DIR / '05_confusion_matrix_lgbm_v2.png'
    plot_confusion_matrix(
        y_test, y_pred_lgbm, CLASS_NAMES,
        'LightGBM v2 — 3 clases', cm_path
    )
    mlflow.log_artifact(str(cm_path))
    mlflow.lightgbm.log_model(best_lgbm, 'lightgbm_v2')
    run_id_lgbm = run_lgbm.info.run_id

print('=== METRICAS EN TEST — LIGHTGBM v2 ===')
print(f'  CV Macro F1 (Optuna) : {study_lgbm.best_value:.4f}')
print(f'  Test Macro F1        : {metricas_lgbm["macro_f1"]:.4f}  <- METRICA PRINCIPAL')
print(f'  Test Kappa           : {metricas_lgbm["kappa"]:.4f}')
print(f'  Test AUC OvR         : {metricas_lgbm["auc_ovr"]:.4f}')
print(f'  Test Accuracy        : {metricas_lgbm["accuracy"]:.4f}')
print(f'\nF1 por clase:')
for cls in CLASS_NAMES:
    print(f'  {cls:<10}: {metricas_lgbm[f"f1_{cls}"]:.4f}')
print(f'\nMLflow run_id: {run_id_lgbm}')


2026/06/05 12:45:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/05 12:45:06 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 12:45:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


=== METRICAS EN TEST — LIGHTGBM v2 ===
  CV Macro F1 (Optuna) : 0.7057
  Test Macro F1        : 0.7166  <- METRICA PRINCIPAL
  Test Kappa           : 0.5773
  Test AUC OvR         : 0.8812
  Test Accuracy        : 0.7182

F1 por clase:
  ALTO      : 0.8015
  BAJO      : 0.7407
  MEDIO     : 0.6077

MLflow run_id: e324f1df2b8849ffaea92ff0265e3884


## 5. XGBoost — Optimización con Optuna

In [5]:
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

def objective_xgb(trial):
    params = {
        'objective'        : 'multi:softprob',
        'num_class'        : N_CLASSES,
        'eval_metric'      : 'mlogloss',
        'verbosity'        : 0,
        'random_state'     : 42,
        'n_estimators'     : trial.suggest_int('n_estimators', 100, 800),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 3, 8),
        'min_child_weight' : trial.suggest_int('min_child_weight', 1, 10),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma'            : trial.suggest_float('gamma', 0, 5),
    }
    model  = xgb.XGBClassifier(**params, use_label_encoder=False)
    scores = []
    for tr_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]
        sw_tr       = sample_weights[tr_idx]
        model.fit(X_tr, y_tr, sample_weight=sw_tr, verbose=False)
        scores.append(f1_score(y_val, model.predict(X_val), average='macro'))
    return np.mean(scores)


print('Ejecutando Optuna para XGBoost (50 trials)...')
study_xgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f'\nMejor macro F1 CV (XGBoost): {study_xgb.best_value:.4f}')
print('Mejores hiperparametros:')
for k, v in study_xgb.best_params.items():
    print(f'  {k:<25}: {v}')


Ejecutando Optuna para XGBoost (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Mejor macro F1 CV (XGBoost): 0.7098
Mejores hiperparametros:
  n_estimators             : 276
  learning_rate            : 0.021302595929384893
  max_depth                : 6
  min_child_weight         : 3
  subsample                : 0.7897655179072602
  colsample_bytree         : 0.904622312421967
  reg_alpha                : 0.13998142462084637
  reg_lambda               : 0.013975316281180504
  gamma                    : 0.6569590194377855


## 6. XGBoost — Entrenamiento final y MLflow

In [6]:
best_xgb = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=N_CLASSES,
    verbosity=0,
    random_state=42,
    use_label_encoder=False,
    **study_xgb.best_params
)

with mlflow.start_run(run_name='xgboost_optuna_v2') as run_xgb:
    mlflow.log_params({
        'model_type'      : 'XGBoost',
        'n_classes'       : N_CLASSES,
        'class_weighting' : 'sample_weight_balanced',
        'optuna_trials'   : 50,
        'target'          : 'TARGET_B_3clases',
        **study_xgb.best_params
    })
    mlflow.log_metric('cv_macro_f1_best', study_xgb.best_value)

    best_xgb.fit(X_train, y_train, sample_weight=sample_weights)
    y_pred_xgb = best_xgb.predict(X_test)
    y_prob_xgb = best_xgb.predict_proba(X_test)

    metricas_xgb = calcular_metricas(y_test, y_pred_xgb, y_prob_xgb, CLASS_NAMES)
    mlflow.log_metrics({f'test_{k}': v for k, v in metricas_xgb.items()})

    cm_path_xgb = FIG_DIR / '06_confusion_matrix_xgb_v2.png'
    plot_confusion_matrix(
        y_test, y_pred_xgb, CLASS_NAMES,
        'XGBoost v2 — 3 clases', cm_path_xgb
    )
    mlflow.log_artifact(str(cm_path_xgb))
    mlflow.xgboost.log_model(best_xgb, 'xgboost_v2')
    run_id_xgb = run_xgb.info.run_id

print('=== METRICAS EN TEST — XGBOOST v2 ===')
print(f'  CV Macro F1 (Optuna) : {study_xgb.best_value:.4f}')
print(f'  Test Macro F1        : {metricas_xgb["macro_f1"]:.4f}')
print(f'  Test Kappa           : {metricas_xgb["kappa"]:.4f}')
print(f'  Test AUC OvR         : {metricas_xgb["auc_ovr"]:.4f}')
print(f'  Test Accuracy        : {metricas_xgb["accuracy"]:.4f}')
print(f'\nF1 por clase:')
for cls in CLASS_NAMES:
    print(f'  {cls:<10}: {metricas_xgb[f"f1_{cls}"]:.4f}')


2026/06/05 12:49:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/05 12:50:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


=== METRICAS EN TEST — XGBOOST v2 ===
  CV Macro F1 (Optuna) : 0.7098
  Test Macro F1        : 0.7107
  Test Kappa           : 0.5699
  Test AUC OvR         : 0.8830
  Test Accuracy        : 0.7132

F1 por clase:
  ALTO      : 0.7912
  BAJO      : 0.7473
  MEDIO     : 0.5938


## 7. Tabla comparativa y modelo seleccionado

In [7]:
metricas_lr = {
    'macro_f1': 0.7107, 'kappa': 0.5661,
    'auc_ovr' : 0.8554, 'accuracy': 0.7107,
}

print('=== TABLA COMPARATIVA — 3 MODELOS (v2) ===')
header = f"{'Metrica':<20} {'LR Baseline':>14} {'LightGBM':>14} {'XGBoost':>14}"
print(header)
print('─' * 65)

for metrica in ['macro_f1', 'kappa', 'auc_ovr', 'accuracy']:
    lr_v   = metricas_lr[metrica]
    lgbm_v = metricas_lgbm[metrica]
    xgb_v  = metricas_xgb[metrica]
    mejor  = max(lr_v, lgbm_v, xgb_v)
    lr_s   = f'{lr_v:.4f}' + (' *' if lr_v   == mejor else '  ')
    lgbm_s = f'{lgbm_v:.4f}' + (' *' if lgbm_v == mejor else '  ')
    xgb_s  = f'{xgb_v:.4f}' + (' *' if xgb_v  == mejor else '  ')
    fila   = f'{metrica:<20} {lr_s:>14} {lgbm_s:>14} {xgb_s:>14}'
    print(fila)

print('─' * 65)
print('* = mejor en esa metrica')

# Seleccion del modelo final
if metricas_lgbm['macro_f1'] >= metricas_xgb['macro_f1']:
    modelo_final = 'LightGBM'
    modelo_obj   = best_lgbm
    metricas_fin = metricas_lgbm
else:
    modelo_final = 'XGBoost'
    modelo_obj   = best_xgb
    metricas_fin = metricas_xgb

mejora = metricas_fin['macro_f1'] - metricas_lr['macro_f1']
print(f'\nMODELO SELECCIONADO: {modelo_final}')
print(f'  Test Macro F1  : {metricas_fin["macro_f1"]:.4f}')
print(f'  Mejora baseline: {mejora:+.4f} ({mejora*100:.1f}pp)')

joblib.dump(modelo_obj, PROCESSED / 'modelo_final.joblib')
print(f'\nModelo guardado en data/processed/modelo_final.joblib')


=== TABLA COMPARATIVA — 3 MODELOS (v2) ===
Metrica                 LR Baseline       LightGBM        XGBoost
─────────────────────────────────────────────────────────────────
macro_f1                   0.7107         0.7166 *       0.7107  
kappa                      0.5661         0.5773 *       0.5699  
auc_ovr                    0.8554         0.8812         0.8830 *
accuracy                   0.7107         0.7182 *       0.7132  
─────────────────────────────────────────────────────────────────
* = mejor en esa metrica

MODELO SELECCIONADO: LightGBM
  Test Macro F1  : 0.7166
  Mejora baseline: +0.0059 (0.6pp)

Modelo guardado en data/processed/modelo_final.joblib
